In [1]:
# !pip install ultralytics

## Imports

In [3]:
import json
import os 
import shutil
from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO
import torch
import matplotlib.pyplot as plt 
from matplotlib import patches
import seaborn as sns 
import random 
from PIL import Image

## Constant

In [5]:
BASE='../CarDD_release/CarDD_COCO'
OUTPUT_BASE = './CarDD_YOLO'
SPLITS = {
    'train' : f'{BASE}/annotations/instances_train2017.json',
    'val'   : f'{BASE}/annotations/instances_val2017.json',
    'test'  : f'{BASE}/annotations/instances_test2017.json',
}
IMG_DIRS = {
    'train' : f'{BASE}/train2017',
    'val'   : f'{BASE}/val2017',
    'test'  : f'{BASE}/test2017',
}

CLASSES= {0: 'dent',
 1: 'scratch',
 2: 'crack',
 3: 'glass shatter',
 4: 'lamp broken',
 5: 'tire flat'}

## COCO to YOLO

In [7]:
for split in SPLITS:
    Path(f"{OUTPUT_BASE}/images/{split}").mkdir(parents=True, exist_ok=True)
    Path(f"{OUTPUT_BASE}/labels/{split}").mkdir(parents=True, exist_ok=True)
    

In [8]:
def coco_to_yolo(json_path, split):
    with open (json_path) as f:
        data = json.load(f)
    img_lookup = {img['id']: img for img in data['images']}
    print(f'len of omg_lookup: {len(img_lookup)}')
    anns_by_images = defaultdict(list)
    for ann in data['annotations']:
        anns_by_images[ann['image_id']].append(ann)
    print(f'len of ann :{len(anns_by_images)} ')
    converted = 0
    for img_id, img_info in img_lookup.items():
        W = img_info['width']
        H = img_info['height']
        fname = img_info['file_name']
        src = os.path.join(IMG_DIRS[split], fname)
        dst = os.path.join(OUTPUT_BASE, 'images', split, fname)
        if not os.path.exists(dst):
            shutil.copy(src, dst)
        label_path = os.path.join(OUTPUT_BASE, 'labels', split,
                                  fname.replace('.jpg', '.txt'))
        lines=[]
        for ann in anns_by_images[img_id]:
            x,y,w,h = ann['bbox']
            x_center = (x+w/2)/W
            y_center = (y+h/2)/H
            w_norm   = w/W
            h_norm   = h/H
            class_id = ann['category_id']-1
            lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} "
                        f"{w_norm:.6f} {h_norm:.6f}")
        with open(label_path, 'w') as f :
            f.write('\n'.join(lines))
        converted +=1
    print(f"  {split}: {converted} images converted")
            
            
        


In [9]:
for split, split_json in SPLITS.items():
   coco_to_yolo(split_json, split) 

len of omg_lookup: 2816
len of ann :2816 
  train: 2816 images converted
len of omg_lookup: 810
len of ann :810 
  val: 810 images converted
len of omg_lookup: 374
len of ann :374 
  test: 374 images converted


In [10]:
print(f"\nOutput structure:")
for split in SPLITS:
    n_imgs   = len(os.listdir(f'{OUTPUT_BASE}/images/{split}'))
    n_labels = len(os.listdir(f'{OUTPUT_BASE}/labels/{split}'))
    print(f"  {split}: {n_imgs} images, {n_labels} label files")


Output structure:
  train: 2816 images, 2816 label files
  val: 810 images, 810 label files
  test: 374 images, 374 label files


## Ymal file 

In [12]:
yaml_content  = f'''
path: {OUTPUT_BASE}

train: images/train
val: images/val
test: images/test

nc: 6

names:
  0: dent
  1: scratch
  2: crack
  3: glass shatter
  4: lamp broken
  5: tire flat
'''

In [13]:
yaml_path = f'{OUTPUT_BASE}/data.yaml'
with open (yaml_path, 'w') as f:
    f.write(yaml_content)
    

In [14]:
with open(yaml_path, 'r') as f:
    print(f.read())


path: ./CarDD_YOLO

train: images/train
val: images/val
test: images/test

nc: 6

names:
  0: dent
  1: scratch
  2: crack
  3: glass shatter
  4: lamp broken
  5: tire flat



## Train YOLOv8

In [16]:
print(f"CUDA available: {torch.cuda.is_available()}")
# print(f"Device        : {torch.cuda.get_device_name(0)}   ")

CUDA available: True


In [17]:
model = YOLO('yolov8m.pt')

In [ ]:
results = model.train(

    data    = yaml_path,
    epochs  = 100,
    imgsz   = 640,
    batch   = 16,
    device  = 0,
    project = './runs',
    name    = 'car_damage_dec',
    patience = 15,
    save    = True,
    plots   = True,
    verbose = True
)

New https://pypi.org/project/ultralytics/8.4.26 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.5  Python-3.10.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./CarDD_YOLO/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=car_damage_dec2

## Evaluation 

In [ ]:
best_model_path = '/kaggle/working/runs/detect/runs/car_damage_dec/weights/best.pt'
model = YOLO(best_model_path)

In [ ]:
test_results = model.val(
    data   = yaml_path, 
    split  = 'test',
    device = 0,
    plots  =True, 
)

In [ ]:
COLORS  = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2','#937860']
test_img_dir = IMG_DIRS['test']
test_images = os.listdir(test_img_dir)
samples = random.sample(test_images, 8)
samples

In [ ]:
fig, axes = plt.subplots(2,4,figsize=(18,9))
axes = axes.flatten()
for ax, fname in zip(axes,samples):
    img_path = os.path.join(test_img_dir, fname)
    result = model.predict(img_path,verbose=False)[0]
    img = Image.open(img_path).convert('RGB')
    W,H = img.size
    ax.imshow(img)
    for box in result.boxes:
        x1,x2,y1,y2 = box.xyxy[0].cpu().numpy()
        cls_id      = int(box.cls[0])
        cls         = list(CLASSES.values())[cls_id]
        conf        = box.conf[0]
        color       = COLORS[cls_id]
        label       = f"{cls} {conf:.2f} "
        rect        = patches.Rectangle((x1,y1), x2-x1, y2-y1, linewidth = 5 , edgecolor = color, facecolor='none' ) 
        ax.add_patch(rect)
        ax.text(x1, y2+1, label, color='white',fontweight='bold')
    ax.set_title(fname)
    ax.axis('off')
plt.suptitle('YOLOv8 Predictions on Test Images')
plt.tight_layout()
plt.savefig('yolo_predictions.png', dpi = 150, bbox_inches = 'tight')
plt.show()